# 02 — Calidad y preparación de datos

**Responsable:** Ignacio Silva  
**Etapa:** Data Preparation (CRISP-DM)  
**Objetivo:** construir una versión reproducible de una fila por canción, sin modificar la fuente en `data/raw/`.

## Punto de partida y controles

La etapa anterior identificó 114.000 filas, tres celdas nulas en una misma observación, una duración no positiva y repetición de `track_id`. Aquí se verifican esos hallazgos antes de transformar. Las reglas reutilizables viven en `src/spotify_popularity/`; el notebook solo deja evidencia y narrativa.

In [ ]:
from spotify_popularity.analysis.quality import dataset_overview
from spotify_popularity.config import PROJECT_PATHS
from spotify_popularity.data.loader import load_raw_dataset
from spotify_popularity.data.preparation import prepare_modeling_dataset


In [ ]:
raw_data = load_raw_dataset()
overview = dataset_overview(raw_data)
overview


## Decisiones de limpieza

1. Se elimina `Unnamed: 0` solo del resultado derivado, pues es un índice de exportación.
2. Se excluye la única fila sin artista, álbum y nombre de pista, cuya duración es cero. No se imputan textos sin evidencia.
3. Se consolida cada `track_id` en una fila. Los atributos musicales, de álbum y textuales fueron invariantes para las repeticiones; la preparación se detiene si esa condición deja de cumplirse.
4. Se conservan todos los géneros como `track_genres`, con etiquetas únicas, ordenadas y separadas por `|`.
5. En los IDs con valores de popularidad distintos se usa la mediana, para no privilegiar una asignación de género por orden de archivo. La popularidad procesada puede ser decimal.

In [ ]:
clean_data, summary = prepare_modeling_dataset(raw_data)
summary


## Validación del resultado

El resultado esperado conserva 89.740 canciones únicas. Se validan identificadores únicos, ausencia de nulos, duración positiva y géneros presentes antes de persistir el archivo regenerable.

In [ ]:
assert clean_data.shape == (89_740, 20)
assert clean_data['track_id'].is_unique
assert clean_data.isna().sum().sum() == 0
assert clean_data['duration_ms'].gt(0).all()
assert clean_data['track_genres'].str.len().gt(0).all()
clean_data.head()


In [ ]:
PROJECT_PATHS.create_generated_directories()
output_path = PROJECT_PATHS.data_processed / 'spotify_tracks_clean.csv'
clean_data.to_csv(output_path, index=False)
output_path


## Entrega y limitaciones

El archivo `data/processed/spotify_tracks_clean.csv` queda preparado para EDA y modelamiento sin modificar el CSV original. `track_genres` es multietiqueta: si la siguiente etapa expande una canción por género, no debe interpretar esas filas como canciones distintas. La popularidad sigue siendo una fotografía histórica, no una medida actual de Spotify. El detalle de decisiones y métricas queda en `docs/progress/ignacio_silva.md`.